# Validacion: reproducir el Dev Macro F1 oficial (modo blind)

**Objetivo:** confirmar que el checkpoint baseline oficial (`eng_model.pth.tar`, `bert-base-multilingual-cased`) reproduce el **Dev Macro F1 = 0.3435** que reporta ahora el README actualizado de `nerel-ds/NEREL-BIO`.

**Contexto:** el README del baseline fue corregido. El numero antiguo (0.6944) se calculaba evaluando solo contra los pares positivos gold ("modo etiquetado"), y ha sido **retractado**. El numero correcto (0.3435) se calcula enumerando **todos los pares candidatos** de cada documento de dev (modo "blind") -- exactamente como puntua CodaBench en el test real.

**Que hace este notebook:**
1. Carga `eng_dev_blind.txt` -- 282.364 pares candidatos generados con `prepare_data.py eng-dev-ent.tsv texts/ -o eng_dev_blind.txt` (modo blind, auto-detectado por no tener columna `relation`).
2. Carga el checkpoint baseline (`eng_model.pth.tar`) y predice sobre los 282k pares usando **inferencia por lotes** (OpenNRE trae `model.infer()` instancia a instancia, que en CPU tarda ~23-30h; en GPU con batching baja a minutos).
3. Evalua las predicciones contra el TSV gold real (`eng-dev-rel.tsv`), emparejando por `(document_id, head_span, tail_span)` -- misma logica que `score.py`.
4. Compara el macro F1 obtenido contra el 0.3435 oficial.

**Datos:** todo (`eng_dev_blind.txt`, `eng-dev-rel.tsv`, `rel2id.json`, `eng_model.pth.tar`) esta en el dataset `bionner-data` ya existente (el mismo que usan los notebooks 1A/2) -- no hace falta subir nada nuevo, solo adjuntarlo con "+ Add Input" si no esta ya adjunto en este notebook.

## 1. Setup

In [1]:
# Ejecutar solo la primera vez
!pip install git+https://github.com/thunlp/OpenNRE.git
!pip install torch transformers nltk pandas scikit-learn

  Cloning https://github.com/thunlp/OpenNRE.git to /tmp/pip-req-build-5vei2f3q
  Running command git clone --filter=blob:none --quiet https://github.com/thunlp/OpenNRE.git /tmp/pip-req-build-5vei2f3q
  Resolved https://github.com/thunlp/OpenNRE.git to commit 8e42fd712f2ab01b48a7e7c4cb2bdea45ec6ff9a
  Preparing metadata (setup.py) ... done
  Created wheel for open-nre: filename=open_nre-0.1.1-py3-none-any.whl size=42022 sha256=b25ce7011280dcbc37eecde440370d07000e9a95f691bc0d9df1afe91eec97f3
  Stored in directory: /tmp/pip-ephem-wheel-cache-44ttxzaf/wheels/ea/ef/3b/4c1a3b1e8f2c5bdbe5961c073dd67f822110066ee2a43bb635
Successfully built open-nre


In [2]:
import json
import time
import logging
import os
from collections import defaultdict
from pathlib import Path

import pandas as pd
import torch

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import opennre

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
N_GPUS = torch.cuda.device_count()
print(f"GPUs detectadas: {N_GPUS}")
for i in range(N_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} - {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PyTorch: 2.10.0+cu128
CUDA disponible: True
GPUs detectadas: 2
  GPU 0: Tesla T4 - 15.6 GB
  GPU 1: Tesla T4 - 15.6 GB


## 2. Configuracion

Todo esta en el dataset `bionner-data` (mismo que usan 1A/2). Si al ejecutar `!ls /kaggle/input` el nombre de carpeta no es `bionner-data`, ajusta `DATA_DIR` para que coincida.

In [3]:
# ============================================================
# CONFIGURACION
# ============================================================
MODEL_NAME = "bert-base-multilingual-cased"
OFFICIAL_DEV_MACRO_F1 = 0.3435  # numero a reproducir (README oficial, track ingles)

MAX_LENGTH = 256
BATCH_SIZE = 64 * max(1, N_GPUS)  # 64 por GPU (con DataParallel se reparte entre las N_GPUS detectadas)

# Todo vive en el mismo dataset bionner-data
DATA_DIR = Path("/kaggle/input/datasets/lucaespernjj/bionner-data")
REL2ID_PATH = DATA_DIR / "rel2id.json"
BLIND_DEV_PATH = DATA_DIR / "eng_dev_blind.txt"
GOLD_TSV_PATH = DATA_DIR / "eng-dev-rel.tsv"
CKPT_PATH = DATA_DIR / "eng_model.pth.tar"

OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_PATH = OUTPUT_DIR / "eng_pred_blind_official.tsv"
RESULTS_PATH = OUTPUT_DIR / "results_validate_blind_official.json"

print(f"BATCH_SIZE efectivo: {BATCH_SIZE} ({N_GPUS} GPU(s))")
for p in [REL2ID_PATH, BLIND_DEV_PATH, GOLD_TSV_PATH, CKPT_PATH]:
    print(f"{'OK ' if p.exists() else 'FALTA '} {p}")

BATCH_SIZE efectivo: 128 (2 GPU(s))
OK  /kaggle/input/datasets/lucaespernjj/bionner-data/rel2id.json
OK  /kaggle/input/datasets/lucaespernjj/bionner-data/eng_dev_blind.txt
OK  /kaggle/input/datasets/lucaespernjj/bionner-data/eng-dev-rel.tsv
OK  /kaggle/input/datasets/lucaespernjj/bionner-data/eng_model.pth.tar


## 3. Cargar datos

In [4]:
def load_instances(path: Path) -> list[dict]:
    instances = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                instances.append(json.loads(line))
    return instances


with open(REL2ID_PATH) as f:
    rel2id = json.load(f)

blind_instances = load_instances(BLIND_DEV_PATH)
gold_df = pd.read_csv(GOLD_TSV_PATH, sep="\t")

print(f"Clases: {len(rel2id)}")
print(f"Candidatos blind: {len(blind_instances)}")
print(f"Relaciones gold reales: {len(gold_df)}")
print(f"Documentos distintos en blind: {len(set(i['doc_id'] for i in blind_instances))}")

Clases: 15
Candidatos blind: 282364
Relaciones gold reales: 2891
Documentos distintos en blind: 50


## 4. Cargar el checkpoint baseline

In [5]:
encoder = opennre.encoder.BERTEntityEncoder(
    max_length=MAX_LENGTH,
    pretrain_path=MODEL_NAME,
)

model = opennre.model.SoftmaxNN(
    sentence_encoder=encoder,
    num_class=len(rel2id),
    rel2id=rel2id,
)

ckpt = torch.load(str(CKPT_PATH), map_location="cpu")
model.load_state_dict(ckpt["state_dict"])
model = model.to(DEVICE)
model.eval()

# Con >1 GPU, envolvemos en DataParallel para repartir cada batch entre todas.
# id2rel se guarda del modelo base (DataParallel no expone los atributos custom directamente).
id2rel = model.id2rel
if N_GPUS > 1:
    infer_model = torch.nn.DataParallel(model)
    print(f"DataParallel activo sobre {N_GPUS} GPUs")
else:
    infer_model = model

print(f"Checkpoint cargado: {CKPT_PATH}")
print(f"Device: {DEVICE}")
print(f"Parametros: {sum(p.numel() for p in model.parameters()):,}")

2026-07-10 14:30:01,066 - root - INFO - Loading BERT pre-trained checkpoint.
2026-07-10 14:30:01,257 - huggingface_hub.utils._http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

DataParallel activo sobre 2 GPUs
Checkpoint cargado: /kaggle/input/datasets/lucaespernjj/bionner-data/eng_model.pth.tar
Device: cuda
Parametros: 180,237,327


## 5. Inferencia por lotes

`model.infer()` de OpenNRE procesa una instancia a la vez. Con 282k pares eso es ~23-30h en CPU. `encoder.tokenize()` siempre rellena (`blank_padding=True`) hasta `max_length`, asi que podemos concatenar varias instancias tokenizadas en un solo batch y llamar directamente al modelo -- mismo resultado, mucho mas rapido en GPU.

Con 2 GPUs, `infer_model` es un `torch.nn.DataParallel` que reparte cada batch entre ambas automaticamente (por eso llamamos a `infer_model(*fields)`, nunca a `model.forward(*fields)` directamente -- eso saltaria el reparto y solo usaria una GPU).

In [6]:
def batched_predict(infer_model, id2rel, encoder, instances, batch_size, device):
    infer_model.eval()
    preds, scores = [], []

    with torch.no_grad():
        for start in range(0, len(instances), batch_size):
            batch = instances[start:start + batch_size]
            tokenized = [
                encoder.tokenize({
                    "text": inst["text"],
                    "h": {"pos": inst["h"]["pos"]},
                    "t": {"pos": inst["t"]["pos"]},
                })
                for inst in batch
            ]
            n_fields = len(tokenized[0])
            fields = [
                torch.cat([t[i] for t in tokenized], dim=0).to(device)
                for i in range(n_fields)
            ]

            # OJO: llamar a infer_model(*fields), NO infer_model.forward(*fields) --
            # con DataParallel, .forward() salta el scatter/gather y solo usa 1 GPU.
            logits = infer_model(*fields)
            probs = torch.softmax(logits, dim=-1)
            batch_scores, batch_preds = probs.max(-1)

            preds.extend(batch_preds.tolist())
            scores.extend(batch_scores.tolist())

            done = min(start + batch_size, len(instances))
            if done % (batch_size * 50) == 0 or done == len(instances):
                print(f"  {done}/{len(instances)} instancias predichas", flush=True)

    return [id2rel[p] for p in preds], scores


print("Funcion de inferencia por lotes definida.")

Funcion de inferencia por lotes definida.


In [7]:
start_time = time.time()

pred_labels, pred_scores = batched_predict(infer_model, id2rel, encoder, blind_instances, BATCH_SIZE, DEVICE)

elapsed = time.time() - start_time
print(f"\nInferencia completada en {elapsed/60:.1f} minutos")
print(f"Velocidad: {len(blind_instances)/elapsed:.1f} instancias/s")

  6400/282364 instancias predichas
  12800/282364 instancias predichas
  19200/282364 instancias predichas
  25600/282364 instancias predichas
  32000/282364 instancias predichas
  38400/282364 instancias predichas
  44800/282364 instancias predichas
  51200/282364 instancias predichas
  57600/282364 instancias predichas
  64000/282364 instancias predichas
  70400/282364 instancias predichas
  76800/282364 instancias predichas
  83200/282364 instancias predichas
  89600/282364 instancias predichas
  96000/282364 instancias predichas
  102400/282364 instancias predichas
  108800/282364 instancias predichas
  115200/282364 instancias predichas
  121600/282364 instancias predichas
  128000/282364 instancias predichas
  134400/282364 instancias predichas
  140800/282364 instancias predichas
  147200/282364 instancias predichas
  153600/282364 instancias predichas
  160000/282364 instancias predichas
  166400/282364 instancias predichas
  172800/282364 instancias predichas
  179200/282364 i

## 6. Construir predicciones (formato CodaBench)

In [8]:
rows = []
for inst, rel, score in zip(blind_instances, pred_labels, pred_scores):
    rows.append({
        "document_id": inst["doc_id"],
        "relation": rel,
        "score": score,
        "head_text": inst["h"]["name"],
        "head_span": inst["head_span"],
        "head_type": inst["head_type"],
        "tail_text": inst["t"]["name"],
        "tail_span": inst["tail_span"],
        "tail_type": inst["tail_type"],
    })

pred_df = pd.DataFrame(rows)
total_pred = len(pred_df)
pred_df_export = pred_df[pred_df["relation"] != "no_relation"].drop(columns=["score"])

print(f"Filtradas {total_pred - len(pred_df_export)} predicciones no_relation")
print(f"Relaciones predichas: {len(pred_df_export)}")

pred_df_export.to_csv(PRED_PATH, sep="\t", index=False)
print(f"Guardado en: {PRED_PATH}")

Filtradas 272949 predicciones no_relation
Relaciones predichas: 9415
Guardado en: /kaggle/working/outputs/eng_pred_blind_official.tsv


## 7. Evaluar contra el gold real

Misma logica de emparejamiento que `baseline/score.py`: clave `(document_id, head_span, tail_span)`. Un par gold que no aparezca en las predicciones cuenta como falso negativo; una prediccion sobre un par que no es gold cuenta como falso positivo.

In [9]:
def create_instance_key(doc_id, head_span, tail_span):
    return f"{doc_id}|{head_span}|{tail_span}"


def evaluate_against_gold(pred_df, gold_df):
    gold_relations = {}
    relation_counts = defaultdict(int)
    for _, row in gold_df.iterrows():
        key = create_instance_key(str(row["document_id"]), str(row["head_span"]), str(row["tail_span"]))
        gold_relations[key] = row["relation"]
        relation_counts[row["relation"]] += 1

    all_relations = sorted(set(gold_df["relation"].unique()))

    pred_relations = {}
    for _, row in pred_df.iterrows():
        key = create_instance_key(str(row["document_id"]), str(row["head_span"]), str(row["tail_span"]))
        pred_relations[key] = row["relation"]

    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for key, pred_rel in pred_relations.items():
        if key in gold_relations:
            gold_rel = gold_relations[key]
            if pred_rel == gold_rel:
                tp[pred_rel] += 1
            else:
                fp[pred_rel] += 1
                fn[gold_rel] += 1
        else:
            fp[pred_rel] += 1

    for key, gold_rel in gold_relations.items():
        if key not in pred_relations:
            fn[gold_rel] += 1

    per_relation = {}
    for rel in all_relations:
        precision = tp[rel] / (tp[rel] + fp[rel]) if (tp[rel] + fp[rel]) > 0 else 0
        recall = tp[rel] / (tp[rel] + fn[rel]) if (tp[rel] + fn[rel]) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        per_relation[rel] = {
            "precision": precision, "recall": recall, "f1": f1,
            "support": relation_counts[rel], "tp": tp[rel], "fp": fp[rel], "fn": fn[rel],
        }

    macro_f1 = sum(per_relation[rel]["f1"] for rel in all_relations) / len(all_relations)

    total_tp, total_fp, total_fn = sum(tp.values()), sum(fp.values()), sum(fn.values())
    micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0

    return {
        "per_relation": per_relation,
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "micro_precision": micro_p,
        "micro_recall": micro_r,
        "total_gold": len(gold_relations),
        "total_pred": len(pred_relations),
    }


results = evaluate_against_gold(pred_df_export, gold_df)

print(f"{'Relacion':<25} {'P':>8} {'R':>8} {'F1':>8} {'Soporte':>8}")
print("-" * 60)
for rel in sorted(results["per_relation"].keys()):
    m = results["per_relation"][rel]
    print(f"{rel:<25} {m['precision']:>8.4f} {m['recall']:>8.4f} {m['f1']:>8.4f} {m['support']:>8}")
print("-" * 60)
print(f"\nMacro F1: {results['macro_f1']:.4f}")
print(f"Micro F1: {results['micro_f1']:.4f} (P={results['micro_precision']:.4f}, R={results['micro_recall']:.4f})")
print(f"Gold relations: {results['total_gold']} | Predichas (no no_relation): {results['total_pred']}")

Relacion                         P        R       F1  Soporte
------------------------------------------------------------
ABBREVIATION                0.2570   0.8889   0.3988       72
AFFECTS                     0.2715   0.7778   0.4025      379
ALTERNATIVE_NAME            0.0413   0.1596   0.0656       95
APPLIED_TO                  0.1679   0.4151   0.2391       53
ASSOCIATED_WITH             0.1116   0.7325   0.1937      228
FINDING_OF                  0.2833   0.6145   0.3878       83
HAS_CAUSE                   0.1627   0.6542   0.2606      481
ORIGINS_FROM                0.1850   0.4805   0.2671       77
PART_OF                     0.1618   0.7188   0.2642      230
PHYSIOLOGY_OF               0.3240   0.7175   0.4464      177
SUBCLASS_OF                 0.4836   0.8604   0.6192      738
TO_DETECT_OR_STUDY          0.1667   0.5893   0.2598      112
TREATED_USING               0.2218   0.6932   0.3361       88
USED_IN                     0.1407   0.7179   0.2353       78
---------

## 8. Comparacion contra el numero oficial

In [10]:
print("=" * 60)
print("COMPARACION")
print("=" * 60)
print(f"Dev Macro F1 oficial (README nerel-ds/NEREL-BIO): {OFFICIAL_DEV_MACRO_F1:.4f}")
print(f"Dev Macro F1 reproducido en este notebook:        {results['macro_f1']:.4f}")
print(f"Diferencia:                                        {results['macro_f1'] - OFFICIAL_DEV_MACRO_F1:+.4f}")
print("=" * 60)

if abs(results["macro_f1"] - OFFICIAL_DEV_MACRO_F1) < 0.01:
    print("\nReproducido correctamente (diferencia < 0.01). El checkpoint local coincide con el oficial.")
else:
    print("\nLa diferencia es mayor de 0.01 -- revisar si el checkpoint es exactamente")
    print("el mismo fichero de GitHub Releases, o si hay alguna diferencia en la")
    print("construccion del blind dev (config de filtrado de tipos, version del TSV, etc).")

COMPARACION
Dev Macro F1 oficial (README nerel-ds/NEREL-BIO): 0.3435
Dev Macro F1 reproducido en este notebook:        0.3126
Diferencia:                                        -0.0309

La diferencia es mayor de 0.01 -- revisar si el checkpoint es exactamente
el mismo fichero de GitHub Releases, o si hay alguna diferencia en la
construccion del blind dev (config de filtrado de tipos, version del TSV, etc).


## 9. Guardar resultados

In [11]:
experiment_results = {
    "experiment": "validate_blind_official",
    "model": MODEL_NAME,
    "checkpoint": str(CKPT_PATH),
    "blind_candidates": len(blind_instances),
    "gold_relations": results["total_gold"],
    "macro_f1": results["macro_f1"],
    "micro_f1": results["micro_f1"],
    "official_dev_macro_f1": OFFICIAL_DEV_MACRO_F1,
    "diff_vs_official": results["macro_f1"] - OFFICIAL_DEV_MACRO_F1,
    "per_relation": results["per_relation"],
    "inference_time_minutes": elapsed / 60,
}

with open(RESULTS_PATH, "w") as f:
    json.dump(experiment_results, f, indent=2)

print(f"Resultados guardados en: {RESULTS_PATH}")

Resultados guardados en: /kaggle/working/outputs/results_validate_blind_official.json
